# Train Sokokali's model (Google Colab)

Fine-tunes an open Qwen3 model on Sokokali's private training data (`nyakslamo/sokokali-training`) and pushes the result privately to Hugging Face.

**Before you start:** Runtime → Change runtime type → **T4 GPU** (free) or **L4 / A100** (Colab Pro). Then run the cells in order.

| GPU | Suggested model | Notes |
|---|---|---|
| T4 (free) | `Qwen/Qwen3-4B` | slower; free sessions can disconnect, so checkpoints go to Google Drive and training resumes |
| L4 / A100 (Pro) | `Qwen/Qwen3-8B` | the recommended size |

In [ ]:
# 1. Check the GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# 2. Get the code and install the training libraries (takes a few minutes)
!git clone -q https://github.com/lamsofttech/Sokokali-AI.git /content/Sokokali-AI 2>/dev/null || git -C /content/Sokokali-AI pull -q
%cd /content/Sokokali-AI
!pip install -q -r training/requirements.txt

In [ ]:
# 3. Log in to Hugging Face: paste your WRITE token in the box that appears (it stays in this Colab session only).
#    Tip: or add it once under the key icon (Colab Secrets) as HF_TOKEN.
import os
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get('HF_TOKEN'))
except Exception:
    login()

In [ ]:
# 4. Keep checkpoints on Google Drive so a disconnect doesn't lose progress
from google.colab import drive
drive.mount('/content/drive')
OUTPUT = '/content/drive/MyDrive/sokokali/outputs'
os.makedirs(OUTPUT, exist_ok=True)

# 5. Download the private training data
from huggingface_hub import snapshot_download
snapshot_download('nyakslamo/sokokali-training', repo_type='dataset', local_dir='data/training')
print(open('data/training/summary.json').read())

In [ ]:
# 6. Choose the model size for your GPU
import torch
big_gpu = torch.cuda.get_device_properties(0).total_memory > 20 * 1024**3
BASE = 'Qwen/Qwen3-8B' if big_gpu else 'Qwen/Qwen3-4B'
HUB_REPO = 'nyakslamo/sokokali-qwen3-8b' if big_gpu else 'nyakslamo/sokokali-qwen3-4b'
EPOCHS = 2 if big_gpu else 1
print(torch.cuda.get_device_name(0), '->', BASE, '->', HUB_REPO, f'({EPOCHS} epoch)')

In [ ]:
# 7. Optional: a quick trial (about 50 steps) to check everything works before the full run
!python training/finetune_qlora.py --train data/training/train.jsonl --eval '' \
    --base-model {BASE} --output /content/trial --epochs 1 --max-train-pairs 400

In [ ]:
# 8. Full training: pushes the adapter privately to Hugging Face when done.
#    If Colab disconnects, re-run cells 2-6 and this one: --resume continues from the last checkpoint on Drive.
!python training/finetune_qlora.py --train data/training/train.jsonl --eval data/training/eval.jsonl \
    --base-model {BASE} --output {OUTPUT} --epochs {EPOCHS} --resume --hub-repo {HUB_REPO}

When cell 8 prints `pushed LoRA adapter to https://huggingface.co/...`, the model is saved privately on Hugging Face.
Tell the Sokokali team (or Claude in VS Code) and the next step is evaluating it on the held-out questions before it goes on air.